# Pauli-transfer-matrix view of the DME/DBAC unitary

A DBAC step is implemented on hardware as a two-qubit unitary built out of multicolor
driven exchange interactions. This notebook visualizes that unitary as a
**Pauli-transfer matrix (PTM)**: the $16\times16$ real matrix $T_{ij} = \tfrac12\mathrm{Tr}[P_j\,U P_i U^\dagger]$
that describes how the unitary $U$ maps each two-qubit Pauli operator $P_i$ onto a
combination of Pauli operators $P_j$. A PTM is a convenient, measurement-friendly way to
compare an ideal unitary against the one actually realized on a device (not done here,
since we only compute ideal unitaries, but the same `unitary_to_ptm`/`plot_ptm` functions
apply to a unitary reconstructed from process tomography).

We compare:

- the **ideal $ZZ(\\theta)$ interaction**, the simplest two-qubit entangling gate;
- the **DME unitary** built out of $ZZ$, $XX$ and $YY$ interactions of the same angle
  (`dme_heisenberg`), which approximates the Heisenberg exchange used on the multicolor
  driven device; and
- **Circuit A** and **Circuit B**, the two concrete DME-based circuits used in the
  experiment, each a DME unitary followed by a single-qubit $R_Z$ calibration correction.

The physics (unitary definitions) and the PTM computation live in
[`scripts/dme_ptm_plot.py`](../scripts/dme_ptm_plot.py); the heatmap plotting lives in
[`visualization/process_matrix.py`](../visualization/process_matrix.py).

In [ ]:
%matplotlib inline
import sys
sys.path.append('..')

import numpy as np
from visualization import use_paper_style, pauli_labels_2q, plot_ptm, plot_ptm_sweep
from scripts.dme_ptm_plot import zz_unitary, dme_heisenberg, rz, unitary_to_ptm
from qutip import Qobj

use_paper_style(font_size=14, figsize=(10, 6))

theta_list = [0, np.pi / 8, np.pi / 4, np.pi / 2]
theta_names = ['0', r'$\pi/8$', r'$\pi/4$', r'$\pi/2$']

## Ideal $ZZ(\theta)$ interaction, swept over $\theta$

In [ ]:
ideal_ptms = [unitary_to_ptm(Qobj(zz_unitary(theta))) for theta in theta_list]
plot_ptm_sweep(ideal_ptms, [f"Ideal PTM (ZZ({name}))" for name in theta_names], pauli_labels_2q);

## DME unitary (ZZ + XX + YY exchange), swept over $\theta$

In [ ]:
dme_ptms = [unitary_to_ptm(Qobj(dme_heisenberg(theta))) for theta in theta_list]
plot_ptm_sweep(dme_ptms, [f"Ideal PTM (DME({name}))" for name in theta_names], pauli_labels_2q);

## Circuits A and B

Each circuit is a DME unitary at a fixed angle, followed by a single-qubit $R_Z$
correction that calibrates out an unwanted phase picked up on the data qubit.

In [ ]:
circuit_a = Qobj(dme_heisenberg(np.pi / 4) @ np.kron(np.eye(2), rz(-0.79)))
plot_ptm(unitary_to_ptm(circuit_a), pauli_labels_2q, title=r"Circuit A (DME($\pi/4$))");

In [ ]:
circuit_b = Qobj(dme_heisenberg(np.pi / 8) @ np.kron(np.eye(2), rz(-np.pi / 4)))
plot_ptm(unitary_to_ptm(circuit_b), pauli_labels_2q, title=r"Circuit B (DME($\pi/8$))");

---
To regenerate all the figures in `plots/` from the command line instead of this notebook, run:

```bash
python scripts/dme_ptm_plot.py
```